In [5]:
import os
import re
import nltk
import pandas as pd
from itertools import combinations
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
FOLDER_PATH="assignments"
THRESHOLD = 60

nltk.download("punkt")
nltk.download("punkt_tab")
documents={}
print("\nLoading assignment documents")
print("-"*60)
for filename in os.listdir(FOLDER_PATH):
    if filename.endswith(".txt"):
        file_path=os.path.join(FOLDER_PATH,filename)
        with open(file_path,"r",encoding="utf-8") as file:
            text=file.read()
        documents[filename]=text
        print("Loaded:",filename)

print("\nTotal documents:",len(documents))
def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove punctuation and special characters
    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()
cleaned_documents={}
for filename,text in documents.items():
    cleaned_documents[filename]=clean_text(text)
print("\nText cleaning completed")
tokenized_documents={}
for filename,text in cleaned_documents.items():
    tokens=nltk.word_tokenize(text)
    tokenized_documents[filename]=tokens
print("Tokenization completed")
print("\nToken counts:")
for filename,tokens in tokenized_documents.items():
    print(filename,"->",len(tokens),"words")
file_names=list(cleaned_documents.keys())
texts=[cleaned_documents[file] for file in file_names]
vectorizer=TfidfVectorizer()
tfidf_matrix=vectorizer.fit_transform(texts)
print("\nTF-IDF matrix:",tfidf_matrix.shape)
similarity_matrix=cosine_similarity(tfidf_matrix)
similarity_matrix=similarity_matrix*100
print("\n")
print("="*80)
print("SIMILARITY MATRIX")
print("="*80)
matrix_df=pd.DataFrame(similarity_matrix,index=file_names,columns=file_names)
print(matrix_df.round(2))
results=[]
for i,j in combinations(range(len(file_names)),2):
    d1=file_names[i]
    d2=file_names[j]
    similarity=similarity_matrix[i][j]
    if similarity>=THRESHOLD:
        status="POSSIBLE PLAGARISM"
    else:
        status="LOW SIMILARITY"
    results.append({"Document 1":d1,"Document 2":d2,"Similarity (%)":similarity,"Status":status})
print("\n")
print("="*80)
print("Document comparison report")
print("="*80)
for result in results:
    print(f"\n{result['Document 1']}"f"vs "f"{result['Document 2']}")
    print(f"Similarity:"f"{result['Similarity (%)']:.2f}%")
    print(f"Status:"f"{result['Status']}")
suspicious_pairs=[result for result in results
                  if result["Similarity (%)"]>=THRESHOLD]

suspicious_pairs.sort(key=lambda x: x["Similarity (%)"],
    reverse=True)

print("\n")
print("="*80)
print("Ranked suspicious document pairs")
print("="*80)
if len(suspicious_pairs)==0:
    print("\nNo suspicious document pair found")
else:
    for rank,result in enumerate(suspicious_pairs,start=1):
        print(f"\n{rank}." f"{result['Document 1']}",f"<->" f"{result['Document 2']}")
        print(f"Similarity: "f"{result['Similarity (%)']:.2f}%")
        print(f"Status: "f"{result['Status']}")
report_df=pd.DataFrame(results)
report_df=report_df.sort_values(by="Similarity (%)",ascending=False)
report_df.to_csv("plagarism_report.csv",index=False)
print("\n")
print("="*80)
print("Report saved")
print("="*80)
print("File:plagarism_report.csv")


Loading assignment documents
------------------------------------------------------------
Loaded: assignment_A.txt
Loaded: assignment_B.txt
Loaded: assignment_C.txt
Loaded: assignment_D.txt
Loaded: assignment_E.txt
Loaded: assignment_F.txt
Loaded: assignment_G.txt
Loaded: assignment_H.txt

Total documents: 8

Text cleaning completed
Tokenization completed

Token counts:
assignment_A.txt -> 59 words
assignment_B.txt -> 69 words
assignment_C.txt -> 51 words
assignment_D.txt -> 41 words
assignment_E.txt -> 43 words
assignment_F.txt -> 49 words
assignment_G.txt -> 44 words
assignment_H.txt -> 49 words

TF-IDF matrix: (8, 158)


SIMILARITY MATRIX
                  assignment_A.txt  assignment_B.txt  assignment_C.txt  \
assignment_A.txt            100.00             94.26             25.19   
assignment_B.txt             94.26            100.00             26.47   
assignment_C.txt             25.19             26.47            100.00   
assignment_D.txt             56.96             66.95 

[nltk_data] Downloading package punkt to C:\Users\HARINI
[nltk_data]     RAJESH\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\HARINI
[nltk_data]     RAJESH\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
